# Chapter 1 — What Is an Agent, Really?

**Book alignment:** current Chapter 1 · internal demo `Stage 00`

The shared demo package calls this **Stage 00** internally. The notebook number follows the book chapter number; the internal stage number is one lower.

**Question this notebook isolates:** Does a changed environment observation alter the next decision?


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "demo" / "agents-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing demo/agents-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / "demo" / "agents-from-first-principles"
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent import Agent, RepositoryEnvironment, Stage00Policy
from first_principles_agent.actions import Action, ActionKind, Observation

## Run against the deliberately broken parser

The target repository contains one passing comma-delimiter test and one failing pipe-delimiter test. Stage 00 can inspect but cannot edit.


In [ ]:
target = DEMO_ROOT / "examples" / "broken-parser"
agent = Agent(
    policy=Stage00Policy(),
    environment=RepositoryEnvironment(target),
)

state = agent.run("Fix pipe-delimited records")

(
    [(t.step, t.action.kind.value, t.observation.summary) for t in state.transitions],
    state.termination_reason,
)

In [ ]:
assert [t.action.kind for t in state.transitions] == [
    ActionKind.RUN_TESTS,
    ActionKind.READ_FILE,
]
assert state.transitions[0].observation.ok is False
assert state.transitions[1].observation.source == "parser.py"
assert state.termination_reason == (
    "inspection complete; repair capability has not been introduced yet"
)

## Causal ablation: change only the first observation

Now keep the policy and loop unchanged, but return a passing test observation. If the system is genuinely adaptive, it should **not** inspect `parser.py`.


In [ ]:
class PassingEnvironment:
    def execute(self, action: Action) -> Observation:
        if action.kind == ActionKind.RUN_TESTS:
            return Observation(
                source="pytest",
                ok=True,
                summary="tests passed",
                details="synthetic control observation",
            )
        raise AssertionError(f"unexpected action: {action.kind}")


control = Agent(Stage00Policy(), PassingEnvironment()).run("Inspect repository")

(
    [(t.action.kind.value, t.observation.summary) for t in control.transitions],
    control.termination_reason,
)

In [ ]:
assert [t.action.kind for t in control.transitions] == [ActionKind.RUN_TESTS]
assert control.termination_reason == "no further Stage 00 action is justified"

print("Observed failure → inspect source")
print("Observed success → stop")

## What we earned

This is not a useful coding agent yet. It cannot repair the repository, validate model output, plan, remember, search, or verify success.

It establishes only the first causal property: **environment observations can change future control flow**.

Notebook 02 / Chapter 2 keeps this control loop and adds the explicit action boundary: model output becomes a proposal that must earn execution authority.
